In [2]:
import polars as pl
import polars.selectors as cs
import seaborn as sns
import matplotlib.pyplot as plt
#global settings
pl.Config.set_engine_affinity("streaming")
#pl.Config.set_tbl_rows(-1)


polars.config.Config

In [3]:
df = pl.read_parquet("../data/parquets/sold_listings_20260830.parquet")
print(df.schema)
print(df.head())

Schema({'title': String, 'department': String, 'source': String, 'category': String, 'category_path': String, 'category_size': String, 'color': String, 'condition': String, 'size': String, 'styles': String, 'country_of_origin': String, 'price': Int32, 'sold_price': Int32, 'created_at': Datetime(time_unit='us', time_zone='UTC'), 'sold_at': Datetime(time_unit='us', time_zone='UTC'), 'cover_photo_url': String, 'location': String, 'seller_id': Int32, 'seller_total_transactions': Int32, 'seller_trusted': Boolean, 'seller_rating_average': Float64, 'seller_rating_count': Int32, 'followers_count': Int32, 'heat_score': Float64, 'photo_count': Int32, 'measurement_count': Int32, 'external_id': Int64, 'currency': String, 'local_image_path': String, 'image_download_status': String, 'original_price': Int32, 'scraped_at': Datetime(time_unit='us', time_zone='UTC'), 'designer_ids': List(Int32), 'designer_names': List(String), 'id': Int64})
shape: (5, 35)
┌─────────────┬────────────┬─────────┬──────────

In [4]:
#Null percentage count by column
nulls = (df.select(pl.all().null_count() / pl.len() * 100)
         .unpivot(variable_name="column_name", value_name="null_percentage")
         .filter(pl.col("null_percentage") > 0)
         .sort("null_percentage", descending=True))
print(nulls)


shape: (8, 2)
┌───────────────────────┬─────────────────┐
│ column_name           ┆ null_percentage │
│ ---                   ┆ ---             │
│ str                   ┆ f64             │
╞═══════════════════════╪═════════════════╡
│ local_image_path      ┆ 100.0           │
│ image_download_status ┆ 100.0           │
│ country_of_origin     ┆ 85.322099       │
│ measurement_count     ┆ 54.85585        │
│ original_price        ┆ 46.708723       │
│ seller_rating_average ┆ 2.976613        │
│ color                 ┆ 2.221885        │
│ condition             ┆ 0.000207        │
└───────────────────────┴─────────────────┘


In [5]:
unique = ((df.select(pl.all().n_unique())
          .unpivot(variable_name="column_name", value_name="unique_count")
          .filter(pl.col("unique_count") > 0))
          .sort("unique_count", descending=True))
print(unique)

shape: (35, 2)
┌───────────────────────┬──────────────┐
│ column_name           ┆ unique_count │
│ ---                   ┆ ---          │
│ str                   ┆ u32          │
╞═══════════════════════╪══════════════╡
│ external_id           ┆ 4345274      │
│ id                    ┆ 4345274      │
│ sold_at               ┆ 4345192      │
│ created_at            ┆ 4344577      │
│ heat_score            ┆ 4316844      │
│ …                     ┆ …            │
│ source                ┆ 1            │
│ currency              ┆ 1            │
│ local_image_path      ┆ 1            │
│ image_download_status ┆ 1            │
│ scraped_at            ┆ 1            │
└───────────────────────┴──────────────┘


In [6]:
#Checks how many listings have original_price == sold_price. original_price is derived from price_drops list so
og_is_sold = df.select(pl.col('original_price'), pl.col('sold_price')).filter(pl.col("sold_price") == pl.col('original_price')).count()
print(og_is_sold)


shape: (1, 2)
┌────────────────┬────────────┐
│ original_price ┆ sold_price │
│ ---            ┆ ---        │
│ u32            ┆ u32        │
╞════════════════╪════════════╡
│ 22444          ┆ 22444      │
└────────────────┴────────────┘


In [7]:
#How many listings have 1, 2 or 3 designers
(((df.select(pl.col('designer_ids').list.len().alias('n'))
 .group_by('n'))
 .len())
 .sort('n'))

n,len
u32,u32
1,2643863
2,1388803
3,312608


In [8]:
df.select(pl.col('designer_names').explode().alias('d')).group_by('d').len().sort('len', descending=True).head(10)

C:\Users\mononoaware\AppData\Local\Temp\ipykernel_28876\3800829800.py:1: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  df.select(pl.col('designer_names').explode().alias('d')).group_by('d').len().sort('len', descending=True).head(10)


d,len
str,u32
"""Vintage""",1051158
"""Streetwear""",549519
"""Nike""",317452
"""Japanese Brand""",218105
"""Supreme""",194381
"""Carhartt""",105500
"""Adidas""",101485
"""Band Tees""",90469
"""Jordan Brand""",87680


In [9]:
df.select(cs.numeric() == 0).sum()


price,sold_price,seller_id,seller_total_transactions,seller_rating_average,seller_rating_count,followers_count,heat_score,photo_count,measurement_count,external_id,original_price,id
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,8,0,129342,288221,0,2383637,1008093,0,0,0


In [10]:
df.select(pl.col('designer_names').list.get(0)).n_unique()
#all_d = df.select(pl.col("designer_names").explode()).unique()

6362

In [11]:
DENY = {
    # seller positioning — consider keeping as boolean flags
    "Rare", "Luxury", "Hype Beast", "Designer", "Streetwear",

    # category descriptors — duplicate category_path
    "Vintage", "Japanese Brand", "Band Tees", "Band T Shirt", "Rap Tees",
    "Archival Clothing", "Soccer Jersey", "Jean",

    # origin / provenance
    "Made In Usa",

    # licensed sports — deny as designer, consider licensed_sports flag
    "MLB", "NFL", "Manchester United",

    # retailers, not manufacturers
    "Hat Club",

    # non-apparel / junk
    "Stickers",

    # catch-all
    "Other",
}

df.filter(pl.col("designer_names").list.get(0).is_in(DENY)).height

505458

In [12]:
(df.select(pl.col("designer_names").list.get(0).alias("d"))
   .group_by("d").len()
   .sort("len", descending=True)
   .head(100))

d,len
str,u32
"""Nike""",255419
"""Supreme""",153081
"""Japanese Brand""",138876
"""Carhartt""",104694
"""Adidas""",101377
…,…
"""Luxury""",8441
"""Raf Simons""",8434
"""True Religion""",8333


In [13]:
DENY = ["Designer", "Luxury", "Rare", "Streetwear", "Stickers", "Made In Usa",
            "Band T Shirt", "Soccer Jersey", "Archival Clothing", "Vintage",
            "Japanese Brand", "Other", "Band Tees", "MLB", "NFL", "Hat Club", "Rap Tees", "Jean", "Hype Beast"]

df.filter(pl.col("designer_names").list.get(0).is_in(DENY)).height

505011

In [14]:
DENY_NORM = {d.lower().strip() for d in DENY}
survivors = pl.col("designer_names").list.eval(
    pl.element().str.to_lowercase().str.strip_chars().is_in(DENY_NORM).not_()
)
df.filter(~survivors.list.any()).height

291747

In [15]:
df.select(pl.col("designer_names").list.first() == '').sum()

designer_names
u32
0


In [16]:
df = df.with_columns(
    pl.col("designer_names")
    .list.eval(
        pl.element().filter(
            pl.element().str.to_lowercase().str.strip_chars().is_in(DENY_NORM).not_()
        )
    )
    .alias('survivors')
)
print(df.filter(pl.col('survivors').list.len() > 1).height)
df = df.with_columns(
    pl.col('survivors').list.first().fill_null("unknown")
)
#df.filter(pl.col('survivors') == 'unknown').height



714343


Restrict the eval window to 2025-08 onward where coverage is at least stable-ish. May be useful later, not in v1.

In [17]:
#First listing w styles was 1stl sold_at 2025
monthly = (df
    .group_by(pl.col('sold_at').dt.truncate('1mo').alias('month'))
    .agg(
        pl.len().alias('total'),
        (pl.col('styles') != '').sum().alias('has_styles'),
    )
    .with_columns((pl.col('has_styles') / pl.col('total')).alias('coverage'))
    .sort('month')
)
with pl.Config(tbl_rows=-1):
    print(monthly)

shape: (56, 4)
┌─────────────────────────┬────────┬────────────┬──────────┐
│ month                   ┆ total  ┆ has_styles ┆ coverage │
│ ---                     ┆ ---    ┆ ---        ┆ ---      │
│ datetime[μs, UTC]       ┆ u32    ┆ u32        ┆ f64      │
╞═════════════════════════╪════════╪════════════╪══════════╡
│ 2021-08-01 00:00:00 UTC ┆ 1      ┆ 0          ┆ 0.0      │
│ 2021-09-01 00:00:00 UTC ┆ 53535  ┆ 0          ┆ 0.0      │
│ 2021-10-01 00:00:00 UTC ┆ 74625  ┆ 0          ┆ 0.0      │
│ 2021-11-01 00:00:00 UTC ┆ 75648  ┆ 0          ┆ 0.0      │
│ 2021-12-01 00:00:00 UTC ┆ 79539  ┆ 0          ┆ 0.0      │
│ 2022-01-01 00:00:00 UTC ┆ 77477  ┆ 0          ┆ 0.0      │
│ 2022-02-01 00:00:00 UTC ┆ 68188  ┆ 0          ┆ 0.0      │
│ 2022-03-01 00:00:00 UTC ┆ 74006  ┆ 0          ┆ 0.0      │
│ 2022-04-01 00:00:00 UTC ┆ 77966  ┆ 0          ┆ 0.0      │
│ 2022-05-01 00:00:00 UTC ┆ 77447  ┆ 0          ┆ 0.0      │
│ 2022-06-01 00:00:00 UTC ┆ 72689  ┆ 0          ┆ 0.0      │
│ 2022-07

Restrict the eval window to 2025-06 -- 08 onward where coverage is at least stable-ish. May be useful later, not in v1.

In [18]:
monthly = (df
    .group_by(pl.col('sold_at').dt.truncate('1mo').alias('month'))
    .agg(
        pl.len().alias('total'),
        (pl.col('country_of_origin') != 'null').sum().alias('has_country'),
    )
    .with_columns((pl.col('has_country') / pl.col('total')).alias('coverage'))
    .sort('month')
)
with pl.Config(tbl_rows=-1):
    print(monthly)

shape: (56, 4)
┌─────────────────────────┬────────┬─────────────┬──────────┐
│ month                   ┆ total  ┆ has_country ┆ coverage │
│ ---                     ┆ ---    ┆ ---         ┆ ---      │
│ datetime[μs, UTC]       ┆ u32    ┆ u32         ┆ f64      │
╞═════════════════════════╪════════╪═════════════╪══════════╡
│ 2021-08-01 00:00:00 UTC ┆ 1      ┆ 0           ┆ 0.0      │
│ 2021-09-01 00:00:00 UTC ┆ 53535  ┆ 0           ┆ 0.0      │
│ 2021-10-01 00:00:00 UTC ┆ 74625  ┆ 0           ┆ 0.0      │
│ 2021-11-01 00:00:00 UTC ┆ 75648  ┆ 0           ┆ 0.0      │
│ 2021-12-01 00:00:00 UTC ┆ 79539  ┆ 0           ┆ 0.0      │
│ 2022-01-01 00:00:00 UTC ┆ 77477  ┆ 0           ┆ 0.0      │
│ 2022-02-01 00:00:00 UTC ┆ 68188  ┆ 0           ┆ 0.0      │
│ 2022-03-01 00:00:00 UTC ┆ 74006  ┆ 0           ┆ 0.0      │
│ 2022-04-01 00:00:00 UTC ┆ 77966  ┆ 0           ┆ 0.0      │
│ 2022-05-01 00:00:00 UTC ┆ 77447  ┆ 0           ┆ 0.0      │
│ 2022-06-01 00:00:00 UTC ┆ 72689  ┆ 0           ┆ 0.0 

In [19]:
df.select(pl.col('location').unique())

location
str
"""Asia"""
"""Europe"""
"""Australia/NZ"""
"""Other"""
"""United States"""
"""Canada"""
"""United Kingdom"""


In [20]:
df = df.with_columns(
        pl.col('category_path').str.extract(r'\.(.+)', 1).alias('path'),
        pl.col('created_at').dt.month().alias('created_month')
    )
df.select(pl.col('path')).filter(pl.col('path').str.contains('.', literal=True)).height

0

In [22]:
df.select(pl.col('sold_at')).sort('sold_at')

sold_at
"datetime[μs, UTC]"
2021-08-02 13:57:29.117 UTC
2021-09-08 18:04:49.561 UTC
2021-09-09 14:20:03.542 UTC
2021-09-09 14:20:10.686 UTC
2021-09-09 14:21:01.850 UTC
…
2026-03-22 11:15:14.233 UTC
2026-03-22 11:25:53.077 UTC
2026-03-22 11:33:19.276 UTC
